# OE3: comprobación reproducible en Google Colab

**Objetivo oficial (literal).** Determinar el efecto de la configuración de los roles de los agentes, los protocolos de comunicación y los mecanismos de inteligencia de enjambre sobre el tiempo de convergencia y la eficiencia de la adaptación de contenidos multimodales.

**Qué comprueba este cuaderno.** Reejecuta el análisis pre-registrado (49 contrastes de efectos principales, ANOVA factorial con η² parcial, Friedman, deriva, sensibilidad y resumen del entorno) desde las observaciones de los 19 200 ciclos y lo compara con el resultado local.

**Datos de entrada.** `raw_observations.csv` (19 200 ciclos), `batches.json`, `environment.json`, `provenance.json`, monitor de recursos y el script de análisis, con el resultado local para comparar.

**Qué no hace.** No recalcula los tiempos medidos, que son los del equipo local con interferencia registrada. No repite las campañas ni regenera datos, no modifica archivos originales, scripts ni umbrales, y no convierte ninguna campaña en controlada. Ejecute las celdas en orden en un entorno nuevo de Colab; la carga de datos pide subir los archivos indicados.

**Contenido (diez pasos).** 1 descripción; 2 datos de entrada; 3 verificación de archivos y estructura; 4 dependencias; 5 ejecución del análisis; 6 reconstrucción de los indicadores; 7 comparación con los resultados oficiales; 8 diferencias numéricas; 9 verificación de las salidas finales; 10 conclusión de reproducibilidad (calculada por la última celda).

In [ ]:
import sys, platform, os
print('Python', sys.version); print(platform.platform())
!pip -q install numpy scipy pandas statsmodels
import numpy, scipy, pandas, statsmodels; print('numpy', numpy.__version__, 'scipy', scipy.__version__, 'pandas', pandas.__version__, 'statsmodels', statsmodels.__version__)
print('CPU', os.cpu_count())

In [ ]:
import zipfile
if not os.path.exists('oe3_colab_paquete_v2.zip'):
    from google.colab import files
    files.upload()
zipfile.ZipFile('oe3_colab_paquete_v2.zip').extractall('oe3')
print(sorted(os.listdir('oe3')), sorted(os.listdir('oe3/campana_01')))

In [ ]:
import hashlib
base='oe3/campana_01'; bad=[]; checked=[]
for l in open(base+'/checksums.txt'):
    l=l.strip()
    if not l: continue
    h,n=l.split(None,1); n=n.strip().lstrip('*'); p=os.path.join(base,n)
    if os.path.exists(p):
        checked.append(n)
        if hashlib.sha256(open(p,'rb').read()).hexdigest()!=h: bad.append(n)
print('verificados:', checked); print('hash distinto:', bad); assert not bad

In [ ]:
!python oe3/analisis_oe3.py oe3/campana_01/raw_observations.csv oe3/salida_colab oe3/campana_01_monitor.csv > oe3/log_analisis.txt 2>&1
print(open('oe3/log_analisis.txt').read()[-600:])

In [ ]:
import json
a=json.load(open('oe3/salida_colab/resultados_oe3.json')); b=json.load(open('oe3/esperado_resultados_oe3_local.json'))
print('verificaciones iguales:', a['verificaciones']==b['verificaciones'], a['verificaciones']['observaciones'], 'observaciones')
mx=0
for ca,cb in zip(a['contrastes'],b['contrastes']):
    assert (ca['respuesta'],ca['factor'],ca['contraste'])==(cb['respuesta'],cb['factor'],cb['contraste'])
    mx=max(mx,abs(ca['estimacion']-cb['estimacion']),abs(ca['ic95'][0]-cb['ic95'][0]),abs(ca['ic95'][1]-cb['ic95'][1]))
print('contrastes comparados:', len(a['contrastes']), '| diferencia maxima en estimaciones e IC:', mx)
mp=max(abs(ca['p_holm']-cb['p_holm']) for ca,cb in zip(a['contrastes'],b['contrastes']))
print('diferencia maxima en p de Holm:', mp)
for k in a['anova_factorial']:
    ea=[(r['termino'],round(r['eta2_parcial'],6)) for r in a['anova_factorial'][k][:6]]; eb=[(r['termino'],round(r['eta2_parcial'],6)) for r in b['anova_factorial'][k][:6]]
    print(k, 'eta2 parcial (6 mayores) iguales:', ea==eb)
print('Friedman iguales:', [round(x['p'],12) for x in a['friedman_tconv']]==[round(x['p'],12) for x in b['friedman_tconv']])
print('despacho/lotes (T_conv):', [ (c['estimacion'],c['ic95']) for c in a['contrastes'] if c['respuesta']=='log_tconv' and c['factor']=='P1_despacho'])
print('sensibilidad:', a['sensibilidad']['despacho_sin_cognitivo_razon_tconv'], b['sensibilidad']['despacho_sin_cognitivo_razon_tconv'])
print('deriva Spearman:', a['deriva']['spearman_residuo_vs_orden'][0], b['deriva']['spearman_residuo_vs_orden'][0])
print('entorno:', a['entorno']['fraccion_con_procesos_ajenos_gt10'], b['entorno']['fraccion_con_procesos_ajenos_gt10'])

## Conclusión de reproducibilidad

In [ ]:
import json, os
def _hojas(x, p=''):
    if isinstance(x, dict):
        for k, v in x.items(): yield from _hojas(v, p+'/'+str(k))
    elif isinstance(x, list):
        for i, v in enumerate(x): yield from _hojas(v, p+'/'+str(i))
    else: yield p, x
def _verif(a, b):
    A = dict(_hojas(json.load(open(a, encoding='utf8')))); B = dict(_hojas(json.load(open(b, encoding='utf8'))))
    mx = 0.0; ng = 0; malas = []
    for k in A:
        if k in B and isinstance(A[k], (int, float)) and isinstance(B[k], (int, float)) and not isinstance(A[k], bool) and not isinstance(B[k], bool):
            d = abs(A[k]-B[k]); mx = max(mx, d)
            if d > 1e-9 and d/max(abs(B[k]), 1e-300) > 1e-9: ng += 1; malas.append((k, A[k], B[k]))
        elif k in B and A[k] != B[k]: ng += 1; malas.append((k, A[k], B[k]))
    return set(A) == set(B), len(A), mx, ng, malas
CHECKS = [('OE3 resultados_oe3.json', 'oe3/salida_colab/resultados_oe3.json', 'oe3/esperado_resultados_oe3_local.json')]
HTTP = []
print('CONCLUSION DE REPRODUCIBILIDAD')
todo_ok = True
dep_version = False
for etiqueta, a, b in CHECKS:
    mismas, n, mx, ng, malas = _verif(a, b)
    solo_wilcoxon = ng > 0 and all(k.endswith('/p_wilcoxon') for k, _, _ in malas)
    ok = mismas and ng == 0; todo_ok &= ok or (mismas and solo_wilcoxon); dep_version |= (not ok) and mismas and solo_wilcoxon
    veredicto = 'COINCIDE' if ok else ('COINCIDE salvo p_wilcoxon (aproximación normal que depende de la versión de scipy)' if mismas and solo_wilcoxon else 'NO COINCIDE')
    print(f'  {etiqueta}: misma estructura {mismas} | hojas {n} | diferencia absoluta máxima {mx:.3e} | hojas con diferencia > 1e-9: {ng} -> {veredicto}')
    for k, x, y in malas[:10]: print(f'     hoja distinta: {k} | Colab {x!r} | local {y!r}')
for k in HTTP:
    r = json.load(open(f'http/out/resumen_{k}.json'))
    ok = r['corridas_validas_finales_recalculadas'] == r['corridas_validas_finales_con_todos_los_campos_iguales'] and max(r['diferencia_abs_max_por_campo'].values()) == 0
    todo_ok &= ok
    print(f"  {k}: indicadores por corrida recalculados desde las peticiones: {r['corridas_validas_finales_recalculadas']} corridas válidas, {r['corridas_validas_finales_con_todos_los_campos_iguales']} idénticas -> {'COINCIDE' if ok else 'NO COINCIDE'}")
if todo_ok and dep_version: print('Resultado: el análisis se reproduce con los datos oficiales; todas las hojas coinciden dentro de 1e-9 salvo los p de Wilcoxon listados arriba, que dependen de la versión de scipy instalada (scipy', __import__('scipy').__version__, 'en este entorno). Las demás cantidades, incluido el ajuste de Holm calculado con p_t, coinciden.')
else: print('Resultado:', 'el análisis se reproduce con los datos oficiales y coincide con el resultado local dentro de la tolerancia de 1e-9 (las diferencias de coma flotante se informan arriba).' if todo_ok else 'HAY DIFERENCIAS: revise las líneas marcadas como NO COINCIDE antes de concluir.')
print('Alcance: se vuelve a ejecutar el análisis estadístico sobre los datos de entrada; no se repiten las campañas ni se regeneran los ciclos o las corridas, y la reproducción no implica que las condiciones de adquisición fueran controladas.')